# EE & Incurred Exploration Across Vehicle Types

This notebook loops through all vehicle types (car, truck, van, suv) and displays:
- Earned Exposure (EE) distributions for all 6 coverages
- Incurred Loss distributions for all 6 coverages
- Zero percentage summary for each

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from config import (
    VEHICLE_TYPES,
    DATA_PATH_TEMPLATE,
    EE_COLUMNS,
    INCURRED_COLUMNS,
    COVERAGES
)

print("Configuration:")
print(f"  Vehicle types: {VEHICLE_TYPES}")
print(f"  Data path template: {DATA_PATH_TEMPLATE}")
print(f"\nEE Columns: {list(EE_COLUMNS.values())}")
print(f"Incurred Columns: {list(INCURRED_COLUMNS.values())}")

## 2. Define Helper Functions

In [ ]:
def load_vehicle_data(vehicle_type):
    """Load data for a specific vehicle type."""
    path = DATA_PATH_TEMPLATE.format(vehicle=vehicle_type)
    
    # Load only the columns we need
    ee_cols = list(EE_COLUMNS.values())
    inc_cols = list(INCURRED_COLUMNS.values())
    columns_to_load = ee_cols + inc_cols
    
    try:
        df = pd.read_parquet(path, columns=columns_to_load)
        return df
    except Exception as e:
        print(f"  ERROR loading {vehicle_type}: {e}")
        return None


def plot_ee_histograms(df, vehicle_type):
    """Plot EE histograms for all coverages."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    for i, (cov, col) in enumerate(EE_COLUMNS.items()):
        ax = axes[i]
        
        if col not in df.columns:
            ax.text(0.5, 0.5, f"Column not found:\n{col}", ha='center', va='center', 
                    transform=ax.transAxes, fontsize=10)
            ax.set_title(f"{cov.upper()} - Missing")
            continue
        
        data = df[col]
        zero_count = (data == 0).sum()
        zero_pct = 100 * zero_count / len(data)
        nonzero_data = data[data > 0]
        
        if len(nonzero_data) > 0:
            ax.hist(nonzero_data, bins=50, edgecolor='black', alpha=0.7)
            ax.set_title(f"{cov.upper()} (Zero: {zero_pct:.1f}%)\nmean={nonzero_data.mean():.4f}")
        else:
            ax.text(0.5, 0.5, f"100% Zero", ha='center', va='center', 
                    transform=ax.transAxes, fontsize=14)
            ax.set_title(f"{cov.upper()} - All Zero!")
        
        ax.set_xlabel(col)
    
    plt.suptitle(f'{vehicle_type.upper()} - Earned Exposure Distributions (Non-Zero Values Only)', fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_incurred_histograms(df, vehicle_type):
    """Plot Incurred Loss histograms for all coverages."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    for i, (cov, col) in enumerate(INCURRED_COLUMNS.items()):
        ax = axes[i]
        
        if col not in df.columns:
            ax.text(0.5, 0.5, f"Column not found:\n{col}", ha='center', va='center', 
                    transform=ax.transAxes, fontsize=10)
            ax.set_title(f"{cov.upper()} - Missing")
            continue
        
        data = df[col]
        zero_count = (data == 0).sum()
        zero_pct = 100 * zero_count / len(data)
        nonzero_data = data[data > 0]
        
        if len(nonzero_data) > 0:
            ax.hist(nonzero_data, bins=50, edgecolor='black', alpha=0.7, color='orange')
            ax.set_title(f"{cov.upper()} (Zero: {zero_pct:.1f}%)\nmean={nonzero_data.mean():.2f}")
        else:
            ax.text(0.5, 0.5, f"100% Zero", ha='center', va='center', 
                    transform=ax.transAxes, fontsize=14)
            ax.set_title(f"{cov.upper()} - All Zero!")
        
        ax.set_xlabel(col)
    
    plt.suptitle(f'{vehicle_type.upper()} - Incurred Loss Distributions (Non-Zero Values Only)', fontsize=14)
    plt.tight_layout()
    plt.show()


def print_zero_summary(df, vehicle_type):
    """Print zero percentage summary for EE and Incurred."""
    print(f"\n--- {vehicle_type.upper()} Zero Percentage Summary ---")
    
    print("\nEarned Exposure:")
    for cov, col in EE_COLUMNS.items():
        if col in df.columns:
            zero_count = (df[col] == 0).sum()
            zero_pct = 100 * zero_count / len(df)
            print(f"  {cov.upper():5s}: {zero_count:>10,} zeros ({zero_pct:>6.2f}%)")
        else:
            print(f"  {cov.upper():5s}: Column not found")
    
    print("\nIncurred Loss:")
    for cov, col in INCURRED_COLUMNS.items():
        if col in df.columns:
            zero_count = (df[col] == 0).sum()
            zero_pct = 100 * zero_count / len(df)
            print(f"  {cov.upper():5s}: {zero_count:>10,} zeros ({zero_pct:>6.2f}%)")
        else:
            print(f"  {cov.upper():5s}: Column not found")

## 3. Loop Through All Vehicle Types

In [ ]:
for vehicle_type in VEHICLE_TYPES:
    print("=" * 70)
    print(f"VEHICLE TYPE: {vehicle_type.upper()}")
    print("=" * 70)
    
    # Load data
    path = DATA_PATH_TEMPLATE.format(vehicle=vehicle_type)
    print(f"\nLoading: {path}")
    
    df = load_vehicle_data(vehicle_type)
    
    if df is None:
        print(f"\nSkipping {vehicle_type} due to loading error.\n")
        continue
    
    print(f"Shape: {df.shape[0]:,} records x {df.shape[1]} columns")
    
    # Plot EE histograms
    plot_ee_histograms(df, vehicle_type)
    
    # Plot Incurred histograms
    plot_incurred_histograms(df, vehicle_type)
    
    # Print summary
    print_zero_summary(df, vehicle_type)
    
    # Free memory
    del df
    
    print("\n")

## Done!

Above you should see EE and Incurred histograms for each vehicle type:
- CAR
- TRUCK
- VAN
- SUV

Check the zero percentages to understand data quality issues across vehicle types.